# पाठ 08 - मल्टी-एजेंट डिज़ाइन पैटर्न


## सेटअप


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## क्यों मल्टी-एजेंट सिस्टम?

वास्तविक दुनिया के कार्य जैसे यात्रा योजना में कई प्रकार की विशेषज्ञता होती है — लॉजिस्टिक्स, स्थानीय ज्ञान, बजट निर्धारण, और भी बहुत कुछ। एक अकेला एजेंट जो सब कुछ संभालने की कोशिश करता है, वह जल्दी ही अव्यवस्थित हो जाता है।

मल्टी-एजेंट सिस्टम इसे **विशेषीकरण** के माध्यम से हल करते हैं: प्रत्येक एजेंट एक विशेषज्ञता के क्षेत्र पर केंद्रित होता है, जो सामान्यज्ञ की तुलना में उच्च गुणवत्ता के परिणाम उत्पन्न करता है। वे **स्केलेबिलिटी** में भी सुधार करते हैं — आप नए एजेंट (जैसे, एक उड़ान विशेषज्ञ, एक रेस्टोरेंट समीक्षक) बिना मौजूदा वर्कफ़्लो को फिर से लिखे जोड़ सकते हैं। एजेंट एक संरचित पाइपलाइन के माध्यम से एक साथ संयोजित होते हैं, संदर्भ को एक से दूसरे तक पास करते हैं।


## विशेषज्ञ एजेंट बनाना


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## एक अनुक्रमिक कार्यप्रवाह बनाना

`WorkflowBuilder` आपको एजेंटों को एक निर्देशित ग्राफ़ में जोड़ने की अनुमति देता है। यहाँ हम एक सरल दो-स्टेप पाइपलाइन बनाते हैं: **TravelPlanner** यात्रा कार्यक्रम तैयार करता है, फिर **TravelConcierge** उसकी समीक्षा करता है और उसे सुधारता है।


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## वर्कफ़्लो में अधिक एजेंट जोड़ना

मल्टी-एजेंट पैटर्न का सबसे बड़ा लाभों में से एक यह है कि इसे बढ़ाना कितना आसान है। नीचे हमने एक **BudgetReviewer** एजेंट जोड़ा है जो योजना की जांच करता है कि वह यात्री के बजट के अनुरूप है या नहीं, उन वस्तुओं को चिह्नित करता है जो लागत को सीमा से ऊपर धकेल सकती हैं, और पैसे बचाने वाले विकल्प सुझाता है। वर्कफ़्लो अब तीन एजेंटों को क्रम में चलाता है:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## सारांश

इस पाठ में आपने सीखा कि कैसे:

1. **विशेषीकृत एजेंट बनाएँ** — प्रत्येक का एक केंद्रित भूमिका होती है (योजना बनाना, कंसीयर्ज, बजट समीक्षा)।
2. **`WorkflowBuilder` और `add_edge` का उपयोग करके एजेंट्स को एक अनुक्रमिक वर्कफ़्लो में जोड़ना**।
3. **मल्टी-एजेंट पाइपलाइन से आउटपुट स्ट्रीम करना**, यह ट्रैक करते हुए कि कौन सा एजेंट बोल रहा है।
4. **वर्कफ़्लो का विस्तार करना** बिना मौजूदा एजेंट्स को बदले, नई एजेंट्स को चेन में जोड़कर।

मल्टी-एजेंट डिजाइन पैटर्न प्रत्येक एजेंट को सरल रखता है जबकि कोई भी एकल एजेंट अकेले हासिल नहीं कर सकता उतनी समृद्ध, अधिक पूरी तरह से समीक्षा की गई परिणाम उत्पन्न करता है।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
